## 10. Evaluation (Quality, Factuality, Faithfulness)
- Automatic metrics:
    - Exact match / F1 for factoid.
    - ROUGE/BLEU for generations (limited value; use cautiously).
    - Constraint satisfaction rate (for structured queries).
    - Faithfulness: string-match (or fuzzy match) of outputs to KG facts.
- Human or rubric-based:
    - Relevance, Completeness, Citation correctness.
- Compare Fine-tuned vs. RAG-KG vs. Zero-shot across a shared test set.
- Summarize in tables/plots.

In [ ]:
import nbimporter
from test1 import rag_kg_baseline, zero_shot_baseline

In [ ]:
import json
import numpy as np
from collections import defaultdict
from pathlib import Path
import re
from difflib import SequenceMatcher
from datasets import load_dataset
import string

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

print("Evaluation libraries loaded")

In [ ]:
# Load test data
test_data = []
with open('data/train/test/val/test_instructions.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        test_data.append(json.loads(line))

print(f"Loaded {len(test_data)} test samples")
print(f"Sample test item: {test_data[0]['instruction']}")


In [ ]:
# RAG-KG predictions on test set
train_value = 1
rag_pred = load_dataset('json', data_files=f'models/baselines/train{train_value}/rag_predictions.jsonl', split='train')
print(rag_pred[0])

In [ ]:
# Zero-shot LLM predictions on test set
zero_pred = load_dataset('json', data_files=f'models/baselines/train{train_value}/zero_predictions.jsonl', split='train')
print(zero_pred[0])

In [ ]:
# Fine-tuned LLM predictions on test set
LLM_pred = load_dataset('json', data_files=f'models/fine_tuned_model/train{train_value}/pred/test_predictions.jsonl', split='train')
print(LLM_pred[0])

In [ ]:
# Extraction of answers from the whole output
def extract_after_inst(row):
    """Extract text after [/INST] and store it in new column"""
    _, _, result = row["model_output"].partition("[/INST]")
    row["model_output_clean"] = result.strip()
    return row

# Apply function to every row
LLM_pred = LLM_pred.map(extract_after_inst)

# Check result
print(LLM_pred[0]["model_output_clean"])

## Automatic metrics

In [ ]:
# Exact Match and F1 Score for FACTOID QA
STOPWORDS = set(stopwords.words('english'))

def normalize_text(text):
    """"""
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    # Tokenize and remove stopwords
    tokens = word_tokenize(text)
    filtered_tokens = [t for t in tokens if t not in STOPWORDS]
    # Join back into a string
    return " ".join(filtered_tokens)

def exact_match(pred, gold):
    """"""
    pred = pred.translate(str.maketrans("","", string.punctuation))
    gold = gold.translate(str.maketrans("","", string.punctuation))
    return int(pred.strip().lower() == gold.strip().lower())

def exact_match_no_stopwords(pred, gold):
    """"""
    return int(normalize_text(pred) == normalize_text(gold))

def f1_score(pred, gold):
    """"""
    pred = pred.translate(str.maketrans("","", string.punctuation))
    gold = gold.translate(str.maketrans("","", string.punctuation))
    pred_tokens = pred.lower().split()
    gold_tokens = gold.lower().split()
    common = set(pred_tokens) & set(gold_tokens)
    if len(common) == 0:
        return 0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def f1_score_no_stopwords(pred, gold):
    """"""
    pred_tokens = normalize_text(pred)
    gold_tokens = normalize_text(gold)
    common = set(pred_tokens) & set(gold_tokens)
    if len(common) == 0:
        return 0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

In [ ]:
# ROUGE score and BLEU score for generations
def rouge_scores(prediction, reference):
    """Compute normalized ROUGE-1/2/L"""
    prediction = prediction.translate(str.maketrans("","", string.punctuation))
    reference = reference.translate(str.maketrans("","", string.punctuation))
    pred_tokens = prediction.strip().lower()
    ref_tokens = reference.strip().lower()

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(ref_tokens, pred_tokens)

    return {
        'rouge1': scores['rouge1'].fmeasure,
        'rouge2': scores['rouge2'].fmeasure,
        'rougeL': scores['rougeL'].fmeasure
    }

def rouge_scores_no_stopwords(prediction, reference):
    """Compute normalized ROUGE-1/2/L"""
    pred_tokens = normalize_text(prediction)
    ref_tokens = normalize_text(reference)

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(ref_tokens, pred_tokens)

    return {
        'rouge1': scores['rouge1'].fmeasure,
        'rouge2': scores['rouge2'].fmeasure,
        'rougeL': scores['rougeL'].fmeasure
    }

def bleu_scores(prediction, reference):
    """Compute BLEU-1/2/3/4 scores"""
    smoothie = SmoothingFunction().method1

    prediction = prediction.translate(str.maketrans("","", string.punctuation))
    reference = reference.translate(str.maketrans("","", string.punctuation))
    pred_tokens = prediction.strip().lower().split()
    ref_tokens = reference.strip().lower().split()
    
    references = [ref_tokens]

    return {
        'bleu1': sentence_bleu(references, pred_tokens, weights=(1, 0, 0, 0), smoothing_function=smoothie),
        'bleu2': sentence_bleu(references, pred_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie),
        'bleu3': sentence_bleu(references, pred_tokens, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoothie),
        'bleu4': sentence_bleu(references, pred_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)
    }

def bleu_scores_no_stopwords(prediction, reference):
    """Compute BLEU-1/2/3/4 scores"""
    smoothie = SmoothingFunction().method1

    pred_tokens = normalize_text(prediction).split()
    ref_tokens = normalize_text(reference).split()
    
    references = [ref_tokens]

    return {
        'bleu1': sentence_bleu(references, pred_tokens, weights=(1, 0, 0, 0), smoothing_function=smoothie),
        'bleu2': sentence_bleu(references, pred_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie),
        'bleu3': sentence_bleu(references, pred_tokens, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoothie),
        'bleu4': sentence_bleu(references, pred_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)
    }

In [ ]:
def constraint_satisfaction_rate(dataset,answer):
    count = 1
    if dataset["constraint_type"] == "quantity_upper":
        numbers = list(map(int, re.findall(r"\d+", answer)))
        target = int(dataset["quantity"])
        if not numbers:
            return 1, count    
        # Must NOT contain any number greater than target
        if all(n <= target for n in numbers):
            return 1, count
        else:
            return 0, count
    if dataset["constraint_type"] == "quantity_lower":
        numbers = list(map(int, re.findall(r"\d+", answer)))
        target = int(dataset["quantity"])
        if not numbers:
            return 1, count    
        # Must NOT contain any number smaller than target
        if all(n >= target for n in numbers):
            return 1, count
        else:
            return 0, count
    if dataset["constraint_type"] == "exact":
        numbers = list(map(int, re.findall(r"\d+", answer)))
        target = int(dataset["quantity"])
        if not numbers and target != 1:
            return 0, 0 ## count=0 così non la conta, è un problema se lista ad esempio i nutrienti con una virgola senza fare l'elenco puntato numerico
        if not numbers:
            return 0, 0 ## count=0 così non la conta, è un problema se lista ad esempio i nutrienti con una virgola senza fare l'elenco puntato numerico                
        if target not in numbers:
            return 0, count   
            # Must NOT contain any number greater than target
        if all(n <= target for n in numbers):
            return 1, count
        else:
            return 0, count
    if dataset["constraint_type"] == "more":
        numbers = list(map(int, re.findall(r"\d+", answer)))
        target = int(dataset["quantity"])
        if not numbers:
            return 0, 0 ## count=0 così non la conta, è un problema se lista ad esempio i nutrienti con una virgola senza fare l'elenco puntato numerico
        if target not in numbers:
            return 0, count
        else:
            return 1, count
    if dataset["constraint_type"] == "less":
        numbers = list(map(int, re.findall(r"\d+", answer)))
        target = int(dataset["quantity"])
        if not numbers:
            return 0, 0 ## count=0 così non la conta, è un problema se lista ad esempio i nutrienti con una virgola senza fare l'elenco puntato numerico
        # Must NOT contain any number greater than target
        if all(n <= target for n in numbers):
            return 1, count
        else:
            return 0, count

### MANCA LA FAITHFULLNESS

## Rubric-based metrics

In [ ]:
import json
from tqdm import tqdm
import os
from groq import Groq

# Initialize client
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY")
)

EVAL_PROMPT = """
You are an expert evaluator of AI-generated answers.

Given:
- the original query,
- and the model response

your job is to score the response according to the following criteria:

1. Relevance (0-1):
   - 1.0 = directly answers the question, stays on topic
   - 0.5 = partially relevant, some drift
   - 0.0 = irrelevant or mostly off-topic

2. Completeness (0-1):
   - 1.0 = fully answers all key parts of the question
   - 0.5 = answers part of the question / missing details
   - 0.0 = largely incomplete

3. Citation Correctness (0-1):
   - 1.0 = all claims that cite sources match the given context
   - 0.5 = some correct, some hallucinated
   - 0.0 = citations do not match context or are hallucinated
   (If no citations are used, score should be 1.0)

Return ONLY valid JSON in this format:
{
  "relevance": float,
  "completeness": float,
  "citation": float
}
"""

def score_with_groq(question, response):
    try:
        chat_completion = client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=[
                {"role": "system", "content": EVAL_PROMPT},
                {"role": "user", "content": f"QUESTION:\n{question}\n\nRESPONSE:\n{response}"}
            ]
        )
        raw = chat_completion.choices[0].message.content
        try:
            return json.loads(raw)
        except:
            print("Invalid JSON from model, raw output:")
            print(raw)
            raise
    except Exception as e:
        raw = f"Error generating response: {e}"
        return raw

## Evaluation

In [ ]:
import os
import json
import numpy as np

def evaluate_all_systems(rag_pred, zero_pred, LLM_pred, test_data, stopwords=True):
    """
    Runs evaluation for 3 answer generation systems on a dataset and returns
    a dict with average scores aggregated across all samples.

    Parameters:
    LLM_pred : list[dict]
        Contains model_output_clean and expected_output for each sample.
    test_data : list[dict]
        Contains instructions and question_type for each sample.
    stopwords : bool (default: True)
        If False → use *_no_stopwords evaluation functions instead.
        If True  → use standard evaluation functions.

    Returns:
    dict : {system_name: {metric_name: average_value}}
    """

    # Choose evaluation functions depending on stopwords flag
    if stopwords:
        em_fn = exact_match
        f1_fn = f1_score
        rouge_fn = rouge_scores
        bleu_fn = bleu_scores
    else:
        em_fn = exact_match_no_stopwords
        f1_fn = f1_score_no_stopwords
        rouge_fn = rouge_scores_no_stopwords
        bleu_fn = bleu_scores_no_stopwords

    # Define expanded metric keys
    METRICS = [
        "exact_match",
        "f1_score",
        "rouge1", "rouge2", "rougeL",
        "bleu1", "bleu2", "bleu3", "bleu4",
        "constraint_score", "constraint_count",
        "groq_relevance", "groq_completeness", "groq_citation"
    ]

    # Prepare result storage
    results = {
        "llm": {metric: [] for metric in METRICS},
        "rag_kg": {metric: [] for metric in METRICS},
        "zero_shot": {metric: [] for metric in METRICS}
    }

    # Main loop
    for i in range(len(test_data)):
        instruction = test_data[i]["instruction"]
        true_answer = LLM_pred[i]["expected_output"]

        # System predictions
        pred_llm = LLM_pred[i]["model_output_clean"]
        pred_ragkg = rag_pred[i]["prediction"]
        pred_zero = zero_pred[i]["prediction"]
        
        systems = {
            "llm": pred_llm,
            "rag_kg": pred_ragkg,
            "zero_shot": pred_zero
        }

        for system_name, prediction in systems.items():

            # Basic metrics
            if test_data[i]["question_type"] == "factoid":
                results[system_name]["exact_match"].append(em_fn(prediction, true_answer))
                results[system_name]["f1_score"].append(f1_fn(prediction, true_answer))

            # Rouge metrics
            rouge = rouge_fn(prediction, true_answer)
            results[system_name]["rouge1"].append(rouge["rouge1"])
            results[system_name]["rouge2"].append(rouge["rouge2"])
            results[system_name]["rougeL"].append(rouge["rougeL"])

            # BLEU metrics
            bleu = bleu_fn(prediction, true_answer)
            results[system_name]["bleu1"].append(bleu["bleu1"])
            results[system_name]["bleu2"].append(bleu["bleu2"])
            results[system_name]["bleu3"].append(bleu["bleu3"])
            results[system_name]["bleu4"].append(bleu["bleu4"])

            # Constraint satisfaction
            if test_data[i]["question_type"] == "constraint":
                score, count = constraint_satisfaction_rate(test_data[i], prediction)
            else:
                score, count = 0.0, 0

            results[system_name]["constraint_score"].append(score)
            results[system_name]["constraint_count"].append(count)

            # Groq evaluator
            groq_dict = score_with_groq(instruction, prediction)
            results[system_name]["groq_relevance"].append(groq_dict["relevance"])
            results[system_name]["groq_completeness"].append(groq_dict["completeness"])
            results[system_name]["groq_citation"].append(groq_dict["citation"])

    # Compute mean values per system
    average_scores = {}

    for system_name, metrics_dict in results.items():
        avg_dict = {}

        # Standard mean metrics (everything except constraint handling)
        for metric, values in metrics_dict.items():
            if metric not in ["constraint_score", "constraint_count"]:
                avg_dict[metric] = float(np.mean(values)) if len(values) > 0 else 0.0

        # Special computation for constraint score
        total_constraint_score = sum(results[system_name]["constraint_score"])
        total_constraint_count = sum(results[system_name]["constraint_count"])

        avg_dict["constraint_score"] = total_constraint_score / total_constraint_count if total_constraint_count > 0 else 0.0

        average_scores[system_name] = avg_dict

    return average_scores


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

def plot_metric_bars(average_scores, train_value, file_ext="png"):
    """
    Plots one grouped bar plot per evaluation metric and saves each figure inside:
        evaluation/plots/train{train_value}/

    Parameters:
    average_scores : dict
        Output of evaluate_all_systems()
    train_value : str or int
        Value used to create folder name, e.g. "1000", "baseline", "v2"
    file_ext : str (png or jpg)
        Output format
    """

    # Build dynamic save folder
    save_folder = os.path.join("evaluation", "plots", f"train{train_value}")
    os.makedirs(save_folder, exist_ok=True)

    systems = list(average_scores.keys())                         # ['llm', 'rag_kg', 'zero_shot']
    metrics = list(next(iter(average_scores.values())).keys())    # metric names

    for metric in metrics:
        values = [average_scores[sys][metric] for sys in systems]
        values = [0.0 if v is None else v for v in values]        # handle None

        x = np.arange(len(systems))

        plt.figure(figsize=(7, 4))
        plt.bar(x, values)
        plt.xticks(x, systems, fontsize=10)
        plt.ylabel(metric.replace("_", " ").title(), fontsize=12)
        plt.title(f"{metric.replace('_', ' ').title()} Comparison", fontsize=14)
        plt.tight_layout()

        # Build filename
        filename = f"evaluation_comparison_{metric}_{train_value}.{file_ext}"
        filepath = os.path.join(save_folder, filename)

        # Save the figure
        plt.savefig(filepath, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"Saved: {filepath}")


In [ ]:
avg_scores = evaluate_all_systems(rag_pred, zero_pred, LLM_pred, test_data, stopwords=True)
plot_metric_bars(avg_scores, train_value=train_value)

## 11. Hallucination Detection & Mitigation
- Detection: For each generated answer:
    - Align claimed entities/values to KG; mark unsupported spans.
    - Hallucination rate = % answers with unsupported claims..

In [ ]:
import json
import os
from itertools import combinations
import matplotlib.pyplot as plt


train_value = 2   # change when running for a different train size
prediction_files = {
    "RAG": f"models/baselines/train{train_value}/rag_predictions.jsonl",
    "Zero-Shot": f"models/baselines/train{train_value}/zero_shot_predictions.jsonl",
    "Fine-Tuned": f"models/fine_tuned_model/train{train_value}/pred/test_predictions.jsonl"
}

output_plot_path = f"evaluation/plots/train{train_value}/hallucination_rate_{train_value}.png"
os.makedirs(os.path.dirname(output_plot_path), exist_ok=True)

entities_file = "data/entities_new.jsonl"
facts_file = "data/facts.jsonl"


# Load entities
entity_dict = {}  # label -> type
with open(entities_file, "r", encoding="utf8") as f:
    for line in f:
        obj = json.loads(line)
        entity_dict[obj["label"]] = obj["type"]

# Load facts
facts_set = set()
with open(facts_file, "r", encoding="utf8") as f:
    for line in f:
        obj = json.loads(line)
        facts_set.add((obj["subject"], obj["predicate"], obj["object"]))


# Predicate logic function
def determine_predicate(t1, t2):
    if t1 == "ingredient" and t2 == "nutrient":
        return "hasNutrient"
    elif t1 == "ingredient" and t2 == "technique":
        return "usesTechnique"
    elif t1 == "ingredient" and t2 == "dietaryGuideline":
        return "hasGuideline"
    elif t1 == "ingredient" and t2 == "healthOutcome":
        return "associatedWithOutcome"
    elif t1 == "ingredient" and t2 == "environmentImpact":
        return "hasEnvironmentalImpact"
    elif t1 == "nutrient" and t2 == "healthOutcome":
        return "affectsRiskOf"
    elif t1 == "dietaryGuideline" and t2 == "technique":
        return "recommendsTechnique"
    elif t1 == "dietaryGuideline" and t2 == "healthOutcome":
        return "aimsToImprove"
    elif t1 == "dietaryGuideline" and t2 == "environmentImpact":
        return "guidelineTargetsImpact"
    elif t1 == "technique" and t2 == "environmentImpact":
        return "affectsImpactCategory"
    return None


def compute_hallucination_rate(jsonl_path):
    if not os.path.exists(jsonl_path):
        print(f"FILE NOT FOUND: {jsonl_path}")
        return None

    total_lines = 0
    hallucinated_lines = 0

    with open(jsonl_path, "r", encoding="utf8") as f:
        for line in f:
            total_lines += 1
            obj = json.loads(line)
            text = obj["prediction"].lower()

            # find entities in prediction text
            found_entities = [{"entity": e, "type": entity_dict[e]}
                              for e in entity_dict if e.lower() in text]

            triples = []
            for e1, e2 in combinations(found_entities, 2):
                pred = determine_predicate(e1["type"], e2["type"])
                if pred:
                    triples.append((e1["entity"], pred, e2["entity"]))

            hallucinated = False
            for t in triples:
                if t not in facts_set:
                    hallucinated = True
                    break

            if hallucinated and triples:
                hallucinated_lines += 1

    return (hallucinated_lines / total_lines) * 100 if total_lines > 0 else 0


# Compute rates for the models
rates = {}
for name, path in prediction_files.items():
    rate = compute_hallucination_rate(path)
    rates[name] = rate
    print(f"{name} hallucination rate: {rate:.2f}%")

# Plot bar chart
labels = list(rates.keys())
values = list(rates.values())

plt.figure(figsize=(6, 4))
plt.bar(labels, values)
plt.ylabel("Hallucination Rate (%)")
plt.title(f"Hallucination Rate Comparison (train={train_value})")
plt.ylim(0, 100)

plt.savefig(output_plot_path, bbox_inches="tight")
print(f"\nPlot saved to: {output_plot_path}")
